# Control difuso de temperatura (Mamdani)
### IA — Unidad III · Sistemas de inferencia difusa y control inteligente
**Autor:** Enzo Fernandez

Controlador difuso **Mamdani** de dos entradas (error `e` y variación del error `Δe`) y una
salida (potencia térmica `u`) que lleva la temperatura de una cámara térmica de `T(0)=20 °C`
a una referencia `Tref` y la mantiene, con respuesta rápida, **sin sobreimpulso excesivo** y
sin usar innecesariamente la máxima potencia.

Lo que se demuestra es la cadena **membresías → reglas → superficie de control → respuesta temporal**.

---
### Decisiones de implementación (puntos abiertos de la guía)
1. **Librería:** implementación propia vectorizada con `numpy` (sin `scikit-fuzzy`), para poder
   parametrizar el ancho/solapamiento de las membresías y re-ejecutar los experimentos con un solo cambio.
2. **Límites de las membresías:** propuestos y dejados como parámetros arriba del código
   (triangulares para los términos centrales, trapezoidales para los extremos).
3. **Tabla de 9 reglas:** completada y justificada (más abajo).
4. **Normalización de la salida:** el actuador trabaja en `uN = u/100 ∈ [−1, 1]`.
5. **Δe(0):** en `k=0` no existe `e(k−1)`; se define `Δe(0)=0`.
6. **Horizonte de simulación:** `t_final = 40 min` (`Δt = 0.02 min` ⇒ 2000 pasos), suficiente para ver establecimiento.
7. **Saturación de Δe:** `Δe` se satura al universo `[−5, 5] °C/min` para que un salto de error no lo desborde.
8. **Banda de establecimiento:** criterio `±0.5 °C` alrededor de `Tref`.
9. **Reproducibilidad:** los experimentos cambian **un solo parámetro** (`c_half` u `ov`) y reusan `simulate()`.

> **Nota de diseño (importante).** La inferencia difusa entrega la **tendencia/incremento** de
> potencia `Δu`, que se **integra** para obtener la potencia del actuador
> `u(k) = clip(u(k−1) + Kdu·Δu(k)·Δt, −100, 100)` (esquema **PI difuso incremental**).
> Esta acción integral es la que anula el error estacionario: un controlador puramente
> proporcional con *acción cero en la referencia* no podría sostener `T > Ta` y dejaría error
> permanente. Con ello, las **mismas reglas** sirven para cualquier `Tref` y el error final ≈ 0.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa
np.set_printoptions(suppress=True)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10,
                     "axes.grid": True, "grid.alpha": 0.3})

## 1. Funciones de membresía (parametrizadas)

Universos: `e ∈ [−10, 10] °C`, `Δe ∈ [−5, 5] °C/min`, `u ∈ [−100, 100] %`.

- **Error** y **Δe**: 3 términos `N` (Negativo), `C` (Cero), `P` (Positivo).
- **Salida**: 5 términos `EF, ES, C, CS, CF` (Enfriamiento Fuerte/Suave, Cero, Calentamiento Suave/Fuerte).

Parámetros clave dejados **arriba** para los experimentos:
- `c_half`: semiancho de la triangular `C` del error (**Experimento 1**).
- `ov`: solapamiento de los hombros `N/P` del error (**Experimento 2**).

In [ ]:
def trimf(x, a, b, c):
    "Triangular: puntos (a,b,c), pico en b."
    x = np.asarray(x, float); y = np.zeros_like(x)
    if b > a:
        i = (x > a) & (x <= b); y[i] = (x[i]-a)/(b-a)
    if c > b:
        i = (x > b) & (x < c);  y[i] = (c-x[i])/(c-b)
    y[x == b] = 1.0
    return np.clip(y, 0, 1)

def trapmf(x, a, b, c, d):
    "Trapezoidal: puntos (a,b,c,d), meseta en [b,c]."
    x = np.asarray(x, float); y = np.zeros_like(x)
    if b > a:
        i = (x > a) & (x < b); y[i] = (x[i]-a)/(b-a)
    y[(x >= b) & (x <= c)] = 1.0
    if d > c:
        i = (x > c) & (x < d); y[i] = (d-x[i])/(d-c)
    return np.clip(y, 0, 1)

# --- Membresias de cada variable (parametrizadas) ---
def error_mfs(e, c_half=4.0, ov=0.0):
    # ov=0 -> particion suma-a-uno (base). ov>0 mayor solapamiento; ov<0 menor.
    N = trapmf(e, -10, -10, -c_half, ov)
    C = trimf(e, -c_half, 0.0, c_half)
    P = trapmf(e, -ov, c_half, 10, 10)
    return {"N": N, "C": C, "P": P}

def de_mfs(de):
    return {"N": trapmf(de, -5, -5, -2, 0),
            "C": trimf(de, -2, 0, 2),
            "P": trapmf(de, 0, 2, 5, 5)}

def u_mfs(u):
    return {"EF": trapmf(u, -100, -100, -80, -50),
            "ES": trimf(u, -75, -40, 0),
            "C":  trimf(u, -25, 0, 25),
            "CS": trimf(u, 0, 40, 75),
            "CF": trapmf(u, 50, 80, 100, 100)}

U = np.linspace(-100, 100, 401)   # universo de salida (fino para el centroide)

In [ ]:
# --- Grafico de las membresias de las TRES variables ---
fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
c3 = ["#2563eb", "#f59e0b", "#dc2626"]
c5 = ["#1d4ed8", "#0891b2", "#64748b", "#ea580c", "#b91c1c"]

e = np.linspace(-10, 10, 601)
for (k, v), c in zip(error_mfs(e).items(), c3):
    ax[0].plot(e, v, c, lw=2, label=k)
ax[0].set_title("Error e [°C]"); ax[0].set_xlabel("e = Tref − T")

d = np.linspace(-5, 5, 601)
for (k, v), c in zip(de_mfs(d).items(), c3):
    ax[1].plot(d, v, c, lw=2, label=k)
ax[1].set_title("Variación del error Δe [°C/min]"); ax[1].set_xlabel("Δe")

uu = np.linspace(-100, 100, 801)
for (k, v), c in zip(u_mfs(uu).items(), c5):
    ax[2].plot(uu, v, c, lw=2, label=k)
ax[2].set_title("Salida u [%] (tendencia de potencia)"); ax[2].set_xlabel("u")

for a in ax:
    a.set_ylim(-0.05, 1.08); a.set_ylabel("μ"); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 2. Base de reglas (9 reglas)

| Error \\ Δe | **N** | **C** | **P** |
|-------------|-------|-------|-------|
| **N**       | EF    | ES    | C     |
| **C**       | ES    | C     | CS    |
| **P**       | C     | CS    | CF    |

**Justificación (recordando que la salida es la *tendencia* de potencia):**
- Diagonal principal `C`: cuando `e` y `Δe` se “compensan” (p. ej. estoy por debajo pero acercándome)
  la acción es neutra: no hace falta cambiar la potencia.
- `e=P, Δe=P` → **CF**: por debajo de la referencia **y alejándose** → subir potencia fuerte.
- `e=N, Δe=N` → **EF**: por encima **y alejándose** → bajar potencia fuerte.
- `e=C, Δe=C` → **C**: en la referencia y estable → mantener (incremento nulo).
- Las casillas intermedias (`ES`, `CS`) dan una corrección suave y hacen la superficie continua.

La matriz es **completa** (9/9), **coherente** (simétrica e impar respecto del origen) y
**físicamente razonable**.

In [ ]:
#            Δe:  N     C     P
RULES = {
    "N": {"N": "EF", "C": "ES", "P": "C"},
    "C": {"N": "ES", "C": "C",  "P": "CS"},
    "P": {"N": "C",  "C": "CS", "P": "CF"},
}

import pandas as pd
tabla = pd.DataFrame(RULES).T
tabla.index.name = "Error \\ Δe"
tabla = tabla[["N", "C", "P"]]
print("Base de reglas (Error x Δe -> salida):")
tabla

## 3. Motor de inferencia Mamdani

- AND → **mínimo**, Implicación → **mínimo**, Agregación → **máximo**, Defuzzificación → **centroide**.

In [ ]:
_OUT = u_mfs(U)   # membresias de salida precalculadas sobre U

def infer(e_val, de_val, c_half=4.0, ov=0.0):
    "Mamdani: (e, Δe) escalares -> u (centroide) en [-100,100]."
    me = {k: float(v[0]) for k, v in error_mfs(np.array([e_val]), c_half, ov).items()}
    md = {k: float(v[0]) for k, v in de_mfs(np.array([de_val])).items()}
    agg = np.zeros_like(U)
    for te in ("N", "C", "P"):
        for td in ("N", "C", "P"):
            w = min(me[te], md[td])              # AND = min
            if w <= 0: continue
            agg = np.maximum(agg, np.minimum(w, _OUT[RULES[te][td]]))  # impl=min, agreg=max
    s = agg.sum()
    return 0.0 if s <= 1e-12 else float((U*agg).sum()/s)   # centroide

# prueba rapida
for ev, dv in [(8, 2), (0, 0), (-8, -2)]:
    print(f"e={ev:+d}, Δe={dv:+d}  ->  u = {infer(ev, dv):+6.1f} %")

In [ ]:
# --- Superficie de control u = f(e, Δe) ---
def control_surface(c_half=4.0, ov=0.0, n=41):
    e_ax = np.linspace(-10, 10, n); de_ax = np.linspace(-5, 5, n)
    Z = np.array([[infer(ev, dv, c_half, ov) for ev in e_ax] for dv in de_ax])
    return e_ax, de_ax, Z

e_ax, de_ax, Z = control_surface()
E, D = np.meshgrid(e_ax, de_ax)
fig = plt.figure(figsize=(6.5, 5))
ax = fig.add_subplot(111, projection="3d")
s = ax.plot_surface(E, D, Z, cmap="coolwarm", antialiased=True)
ax.set_xlabel("e [°C]"); ax.set_ylabel("Δe [°C/min]"); ax.set_zlabel("u [%]")
ax.set_title("Superficie de control base"); ax.view_init(25, -135)
fig.colorbar(s, shrink=0.55, aspect=12, label="u [%]"); plt.tight_layout(); plt.show()

## 4. Modelo de la planta (cámara térmica) y lazo cerrado

Primer orden discreto:
$$T(k{+}1)=T(k)+\Delta t\Big[-\tfrac{T(k)-T_a}{\tau}+\tfrac{K}{\tau}\,u_N(k)\Big],\qquad u_N=u/100.$$

Parámetros: `Ta=20 °C`, `τ=5 min`, `K=15 °C`, `Δt=0.02 min`.
La potencia del actuador se obtiene **integrando** la tendencia difusa (PI difuso incremental).

In [ ]:
def _trap(y, x):
    f = getattr(np, "trapezoid", None) or getattr(np, "trapz", None)  # numpy 2.x / 1.x
    return float(f(y, x))

def metrics(t, T, u, Tref, band=0.5):
    err_final = Tref - T[-1]
    overshoot = max(0.0, T.max() - Tref)
    outb = np.abs(T - Tref) > band
    t_set = t[np.where(outb)[0][-1] + 1] if outb.any() and np.where(outb)[0][-1]+1 < len(t) else (t[-1] if outb.any() else 0.0)
    return dict(err_final=float(err_final), overshoot_C=float(overshoot),
                overshoot_pct=float(100*overshoot/Tref),
                t_set=float(t_set), effort=_trap(np.abs(u), t))

def simulate(Tref=30.0, T0=20.0, Ta=20.0, tau=5.0, K=15.0,
             dt=0.02, t_final=40.0, c_half=4.0, ov=0.0, Kdu=6.0):
    n = int(round(t_final/dt)) + 1
    t = np.arange(n)*dt
    T = np.zeros(n); u = np.zeros(n); du = np.zeros(n)
    T[0] = T0; e_prev = 0.0; u_prev = 0.0
    for k in range(n):
        e = Tref - T[k]
        de = 0.0 if k == 0 else (e - e_prev)/dt      # Δe(0)=0
        de = float(np.clip(de, -5, 5))               # saturar al universo
        du[k] = infer(float(np.clip(e, -10, 10)), de, c_half, ov)   # tendencia difusa
        u[k] = float(np.clip(u_prev + Kdu*du[k]*dt, -100, 100))     # integrador
        e_prev, u_prev = e, u[k]
        if k < n-1:
            T[k+1] = T[k] + dt*(-(T[k]-Ta)/tau + (K/tau)*(u[k]/100.0))
    return dict(t=t, T=T, u=u, du=du, Tref=Tref, metrics=metrics(t, T, u, Tref))

## 5. Respuesta base (`Tref = 30 °C`)

In [ ]:
r = simulate()
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
ax[0].plot(r["t"], r["T"], "#2563eb", lw=2, label="T(t)")
ax[0].axhline(30, color="#dc2626", ls="--", label="Tref = 30 °C")
ax[0].axhline(20, color="gray", ls=":", label="Ta = 20 °C")
ax[0].set_xlabel("t [min]"); ax[0].set_ylabel("T [°C]"); ax[0].set_title("Respuesta temporal"); ax[0].legend(fontsize=8)
ax[1].plot(r["t"], r["u"], "#f59e0b", lw=2); ax[1].axhline(0, color="gray", lw=.8)
ax[1].set_xlabel("t [min]"); ax[1].set_ylabel("u [%]"); ax[1].set_title("Acción de control u(t)"); ax[1].set_ylim(-105, 105)
plt.tight_layout(); plt.show()

m = r["metrics"]
print(f"Error final        : {m['err_final']:+.3f} °C")
print(f"Sobreimpulso máx.  : {m['overshoot_C']:.3f} °C ({m['overshoot_pct']:.2f} %)")
print(f"Tiempo estab. (±0.5): {m['t_set']:.2f} min")
print(f"Esfuerzo ∫|u|dt    : {m['effort']:.0f}")
print(f"Potencia sostén u∞ : {r['u'][-1]:.1f} %  (teórico (Tref-Ta)/K*100 = {(30-20)/15*100:.1f} %)")

## 6. Experimento 1 — Ancho de la función `C` del error

**Predecir:** una `C` más angosta hace al controlador más sensible cerca de la referencia
(pendiente más pronunciada) ⇒ respuesta más rápida; una `C` más ancha lo suaviza ⇒ más lento.
**Modificar → Ejecutar → Observar → Comparar → Explicar** cambiando solo `c_half ∈ {2, 4, 6}`.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
cols = {2: "#2563eb", 4: "#16a34a", 6: "#dc2626"}
e = np.linspace(-10, 10, 601)
for ch in (2, 4, 6):
    ax[0].plot(e, error_mfs(e, c_half=ch)["C"], cols[ch], lw=2, label=f"c_half={ch}")
    ax[1].plot(e, [infer(ev, 0, ch, 0) for ev in e], cols[ch], lw=2, label=f"c_half={ch}")
    r = simulate(c_half=ch)
    ax[2].plot(r["t"], r["T"], cols[ch], lw=2, label=f"c_half={ch}")
    mm = r["metrics"]
    print(f"c_half={ch}: e_final={mm['err_final']:+.3f}  t_set={mm['t_set']:.1f} min  esfuerzo={mm['effort']:.0f}")
ax[0].set_title("(a) Función C"); ax[0].set_xlabel("e"); ax[0].set_ylabel("μ")
ax[1].set_title("(b) Corte de la superficie en Δe=0"); ax[1].set_xlabel("e"); ax[1].set_ylabel("u [%]")
ax[2].set_title("(c) Respuesta temporal"); ax[2].set_xlabel("t [min]"); ax[2].set_ylabel("T [°C]"); ax[2].set_xlim(0, 20)
for a in ax: a.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Observación:** con `c_half=2` la pendiente `du/de` en `e≈0` es mayor ⇒ acción más agresiva ⇒
se establece antes (~5.8 min). Con `c_half=6` la zona cercana a la referencia es más “plana” ⇒
respuesta más lenta (~9.5 min) y algo menos de esfuerzo. El error final es ≈0 en los tres casos:
la acción integral lo corrige; el ancho de `C` afecta el **transitorio**, no el estacionario.

## 7. Experimento 2 — Solapamiento de las funciones del error

**Predecir:** menos solapamiento deja huecos de cobertura (`Σμ<1`) cerca de `e=0` → **zonas de baja
sensibilidad** donde el controlador casi no reacciona; más solapamiento suaviza la superficie.
Se cambia solo `ov ∈ {−2, 0, +2}`.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
lab = {-2: "menor (ov=−2)", 0: "base (ov=0)", 2: "mayor (ov=+2)"}
cols = {-2: "#dc2626", 0: "#16a34a", 2: "#2563eb"}
e = np.linspace(-10, 10, 601)
for ov in (-2, 0, 2):
    m = error_mfs(e, ov=ov)
    ax[0].plot(e, m["N"]+m["C"]+m["P"], cols[ov], lw=2, label=lab[ov])
    ax[1].plot(e, [infer(ev, 0, 4, ov) for ev in e], cols[ov], lw=2, label=lab[ov])
    r = simulate(ov=ov)
    ax[2].plot(r["t"], r["T"], cols[ov], lw=2, label=lab[ov])
    mm = r["metrics"]
    print(f"ov={ov:+d}: e_final={mm['err_final']:+.3f}  t_set={mm['t_set']:.1f} min  esfuerzo={mm['effort']:.0f}")
ax[0].axhline(1, color="gray", ls=":"); ax[0].set_title("(a) Cobertura Σμ"); ax[0].set_xlabel("e"); ax[0].set_ylabel("Σμ")
ax[1].set_title("(b) Corte en Δe=0"); ax[1].set_xlabel("e"); ax[1].set_ylabel("u [%]")
ax[2].set_title("(c) Respuesta temporal"); ax[2].set_xlabel("t [min]"); ax[2].set_ylabel("T [°C]")
for a in ax: a.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Observación:** con `ov=−2` aparece una **zona muerta** cerca de `e=0` (panel b: `u≈0` para
`|e|≲2`). Ahí el incremento difuso es casi nulo, el integrador se estanca y queda **error final ≈2 °C**
(la temperatura se detiene en ~28 °C). Con `ov=+2` la superficie es más suave y la respuesta un poco
más rápida. Conclusión: **solapamiento insuficiente = pérdida de controlabilidad** cerca de la referencia.

## 8. Prueba de cambio de referencia (`Tref = 25 °C`, mismas reglas)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for Tref, c in [(30, "#2563eb"), (25, "#dc2626")]:
    r = simulate(Tref=Tref)
    ax[0].plot(r["t"], r["T"], c, lw=2, label=f"T(t), Tref={Tref}")
    ax[0].axhline(Tref, color=c, ls="--", lw=1)
    ax[1].plot(r["t"], r["u"], c, lw=2, label=f"u(t), Tref={Tref}")
    print(f"Tref={Tref}: T_final={r['T'][-1]:.3f}  e_final={r['metrics']['err_final']:+.3f}  u∞={r['u'][-1]:.1f} %")
ax[0].set_title("Mismas reglas, dos referencias"); ax[0].set_xlabel("t [min]"); ax[0].set_ylabel("T [°C]"); ax[0].legend(fontsize=8)
ax[1].set_title("Acción de control"); ax[1].set_xlabel("t [min]"); ax[1].set_ylabel("u [%]"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

Las reglas operan sobre `e = Tref − T` y `Δe`, **no** sobre `T` absoluta: por eso el mismo juego de
reglas lleva el error a cero tanto para `Tref=30` como para `Tref=25`. La potencia de sostén se ajusta
sola (`u∞ ≈ (Tref−Ta)/K·100`: 66.7 % para 30 °C y 33.3 % para 25 °C).

## 9. Conclusiones
- La cadena membresías → reglas → superficie → respuesta quedó demostrada de punta a punta.
- `Δe` aporta la **tendencia**: para el mismo `e` la acción cambia según si el error crece o decrece.
- El **ancho de `C`** regula la sensibilidad/rapidez cerca de la referencia (transitorio).
- El **solapamiento** garantiza cobertura continua; si es insuficiente aparecen zonas muertas.
- La acción integral (PI difuso incremental) anula el error estacionario y hace las reglas
  independientes de la referencia.
- **Compromiso rapidez ↔ esfuerzo:** acelerar la respuesta (C angosta, más ganancia) exige picos de
  potencia mayores; el diseño base equilibra ambos (sin sobreimpulso, saturación solo en el arranque).